# Osteoarthritis / GWAS pathway enrichment

KEGG-pathway enrichment of osteoarthritis-associated genes, testing whether genes with between-species-divergent cCREs are relevant to within-human disease variation.

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np
import pybedtools
from pybedtools import BedTool
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools
import re
import ast
import math

# --- Shared helpers: const.py lives in analyses/common ---
NB_DIR = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(NB_DIR, '..', 'common')))
import const
from const import pos_active_ctrl_color, neg_active_ctrl_color, highlight_color, custom_cmap
from const import set_equal_plot_limits, plot_color_pallete, custom_cmap_bolder, FONT_SIZE_small
const.set_plot_style()

# --- Paths (EDIT): inputs read / outputs written by this notebook ---
GWAS_RANKING_CSV = '/home/labs/davidgo/noic/hd_project/data/gwas/gwas_gene_ranking/gene_trait_rankings_all_ranked.csv'  # GWAS gene-trait rankings
KEGG_PATHWAY_TSV = '/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Gene_Lists/KEGG/kegg_human_pathway_to_gene_symbol.tsv'  # KEGG pathway -> gene symbol
output_path = '.'  # where plots/tables are written

In [ ]:
gwas_data = pd.read_csv(GWAS_RANKING_CSV)
oa_data = gwas_data[gwas_data['ORIGINAL_TRAIT']=='osteoarthritis']
oa_data_leveled = oa_data[oa_data['rank_level'].isin(['HIGH', 'MEDIUM'])].copy()
oa_genes = oa_data_leveled['Gene_symbol'].unique()

In [ ]:
kegg_data = pd.read_csv(KEGG_PATHWAY_TSV, sep='\t')

In [ ]:
# =====================================================
# KEGG Pathway Enrichment Analysis for OA Genes
# =====================================================

# Step 1: Calculate enrichment statistics for each pathway
enrichment_results = []

for pathway_name in kegg_data['pathway_name'].unique():
    pathway_genes = kegg_data[kegg_data['pathway_name'] == pathway_name]['gene_symbol'].unique()
    total_genes = len(pathway_genes)

    oa_genes_in_pathway = np.intersect1d(pathway_genes, oa_genes)
    oa_genes_count = len(oa_genes_in_pathway)

    percentage = (oa_genes_count / total_genes * 100) if total_genes > 0 else 0

    enrichment_results.append({
        'pathway_name': pathway_name,
        'total_genes': total_genes,
        'oa_genes_count': oa_genes_count,
        'enrichment_percent': percentage
    })

enrichment_df = pd.DataFrame(enrichment_results)
enrichment_df = enrichment_df.sort_values('enrichment_percent', ascending=False).reset_index(drop=True)

enrichment_df['oa_genes_list'] = enrichment_df['pathway_name'].apply(
    lambda pathway: list(np.intersect1d(
        kegg_data[kegg_data['pathway_name'] == pathway]['gene_symbol'].unique(),
        oa_genes
    ))
)

# Step 2: Filter pathways with at least 20 genes for histogram
MIN_GENES = 20
enrichment_df_filtered = enrichment_df[enrichment_df['total_genes'] >= MIN_GENES].copy()

print(f"Pathways with >= {MIN_GENES} genes: {len(enrichment_df_filtered)}")
print(f"Total pathways: {len(enrichment_df)}\n")

# Pathways to highlight
target_pathways = [
    "Glycosaminoglycan biosynthesis - heparan sulfate",
    "Glycosaminoglycan biosynthesis - chondroitin sulfate",
    "Glycosaminoglycan degradation",
    "Glycosaminoglycan biosynthesis - keratan sulfate",
]

# Report target pathways
for name in target_pathways:
    row = enrichment_df[enrichment_df['pathway_name'].str.contains(name, case=False, na=False)]
    if len(row) > 0:
        pct = row['enrichment_percent'].values[0]
        total = row['total_genes'].values[0]
        overlap = row['oa_genes_count'].values[0]
        print(f"{name}")
        print(f"  Total genes: {total}")
        print(f"  OA genes overlap: {overlap}")
        print(f"  Enrichment: {pct:.2f}%\n")
    else:
        print(f"WARNING: '{name}' not found in kegg_data\n")

# Step 3: Create histogram
fig, ax = plt.subplots(figsize=(12, 6))

n, bins, patches = ax.hist(
    enrichment_df_filtered['enrichment_percent'],
    bins=30,
    color='steelblue',
    alpha=0.7,
    edgecolor='black',
    label=f'Pathways (n ≥ {MIN_GENES} genes)'
)

# Add vertical lines for target pathways
colors = ['C1', 'C2', 'C3', 'C4']
for name, col in zip(target_pathways, colors):
    row = enrichment_df[enrichment_df['pathway_name'].str.contains(name, case=False, na=False)]
    if len(row) > 0:
        pct = row['enrichment_percent'].values[0]
        ax.axvline(pct, color=col, linestyle='--', linewidth=2, label=f"{name} ({pct:.2f}%)")

ax.set_xlabel('Enrichment % (OA genes in pathway / Total genes in pathway)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Pathways', fontsize=12, fontweight='bold')
ax.set_title(f'KEGG Pathway Enrichment Distribution\n(Pathways with ≥ {MIN_GENES} genes)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.savefig(output_path + 'kegg_enrichment_histogram.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nHistogram saved to: {output_path}kegg_enrichment_histogram.png")
